In [ ]:
from pathlib import Path
import onnxruntime as ort

ROOT = Path.cwd()
MODEL_DIR = ROOT / "models" / "onnx"

MODEL_FILES = [
    "segnet.onnx",
    "tromr_encoder.onnx",
    "tromr_decoder.onnx",
]

print("Project root:", ROOT)
print("Model dir:", MODEL_DIR)
print("ONNX Runtime:", ort.__version__)

In [ ]:
available = ort.get_available_providers()

print("Available providers:")
for p in available:
    print(" -", p)

if "CUDAExecutionProvider" in available:
    print("\nGPU status: CUDA available")
else:
    print("\nGPU status: CUDA NOT available; ONNX will run on CPU only")

In [ ]:
cuda_options = {
    "device_id": 0,
    "cudnn_conv_algo_search": "HEURISTIC",
    "arena_extend_strategy": "kNextPowerOfTwo",
}

if "CUDAExecutionProvider" in ort.get_available_providers():
    providers = [
        ("CUDAExecutionProvider", cuda_options),
        "CPUExecutionProvider",
    ]
else:
    providers = ["CPUExecutionProvider"]

providers

In [ ]:
for name in MODEL_FILES:
    path = MODEL_DIR / name
    print(f"{name}: {path.exists()}")

    if not path.exists():
        print("  Missing:", path)

In [ ]:
sessions = {}

for name in MODEL_FILES:
    path = MODEL_DIR / name

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    if not path.exists():
        print("SKIPPED: file does not exist")
        continue

    session = ort.InferenceSession(
        str(path),
        providers=providers,
    )

    sessions[name] = session

    print("Active providers:", session.get_providers())

    print("\nInputs:")
    for i in session.get_inputs():
        print(" ", i.name, i.shape, i.type)

    print("\nOutputs:")
    for o in session.get_outputs():
        print(" ", o.name, o.shape, o.type)

In [ ]:
required = set(MODEL_FILES)
loaded = set(sessions.keys())
missing = required - loaded

if not missing:
    print("All ONNX models loaded successfully.")
else:
    print("Missing or failed models:")
    for m in missing:
        print(" -", m)

In [ ]:
cpu_sessions = {}

for name in MODEL_FILES:
    path = MODEL_DIR / name

    if not path.exists():
        continue

    cpu_sessions[name] = ort.InferenceSession(
        str(path),
        providers=["CPUExecutionProvider"],
    )

print("CPU-only sessions loaded:", list(cpu_sessions.keys()))